# Semana 1: higiene e referência corrigida (versão 2, Drive do mestrado)

Esta versão roda inteira no Drive da conta do mestrado. Nenhuma célula treina modelo; tudo roda em CPU, então não precisa de GPU.

**Antes da primeira execução (uma vez só):**

1. **Na conta antiga**, no Drive, selecione as pastas `paper_rs2024_conformal`, `paper_rs2024_resultados`, `paper_rs2024_bolivia` e `prithvi_sen1floods11`, clique em **Compartilhar** e adicione a conta do mestrado como **Leitor**.
2. **Na conta do mestrado**, abra **Compartilhados comigo**, selecione as quatro pastas, clique com o botão direito e escolha **Organizar > Adicionar atalho**. Escolha a pasta `Mestrado/dados`.
3. Abra o Colab **logada na conta do mestrado** (a bolinha no canto superior direito mostra qual conta está ativa) e faça o upload deste notebook lá. Quando a célula 1 pedir permissão, escolha a conta do mestrado.

**O que cada célula faz:**

1. Monta o Drive, confere se é a conta certa e acha os arquivos.
2. Fixa o modelo: confere o hash e, se faltar, baixa a versão exata (commit `918b9f1`) para `Mestrado/modelo_fixado`. Só baixa na primeira vez.
3. Referência da fonte: calibra na val, avalia no test.
4. Compara 16 e 32 bits.
5. Bolívia no τ novo e diferença Bolívia − test, com IC.
6. Salva tudo em `Mestrado/resultados/semana1`.

Rode **na ordem**, uma de cada vez (Shift+Enter). Se alguma célula der erro, pare e cole o erro no Claude.

In [ ]:
# CÉLULA 1. Montar o Drive do mestrado e achar os arquivos
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, re, json, time, hashlib, pickle, shutil
import numpy as np

MEUDRIVE = '/content/drive/MyDrive'
MESTRADO = os.path.join(MEUDRIVE, 'Mestrado')
FIXADO = os.path.join(MESTRADO, 'modelo_fixado')

assert not os.path.isdir(os.path.join(MEUDRIVE, 'DOC old PC')), (
    'Este é o Drive da conta antiga. Vá em Ambiente de execução > Desconectar e excluir ambiente de execução, '
    'abra o Colab logada na conta do mestrado e rode de novo.')
assert os.path.isdir(MESTRADO), f'Não achei a pasta Mestrado. O Meu Drive desta conta tem: {sorted(os.listdir(MEUDRIVE))}'
os.makedirs(FIXADO, exist_ok=True)

ALVOS = {'cal_scores.npy', 'cal_labels.npy', 'cal_chip_sizes.npy', 'cal_chip_names.npy',
         'fonte_registros.pkl', 'bolivia_registros.pkl', 'resultados_bolivia_vs_fonte.json',
         'flood_valid_data.csv', 'flood_test_data.csv', 'config.yaml'}
OBRIGATORIOS = ['cal_scores.npy', 'cal_labels.npy', 'cal_chip_sizes.npy', 'cal_chip_names.npy',
                'fonte_registros.pkl', 'bolivia_registros.pkl', 'resultados_bolivia_vs_fonte.json']
PULAR = {'S1Hand', 'S2Hand', 'LabelHand', 'S1OtsuLabelHand', 'JRCWaterHand', 'S1Weak', 'S2IndexLabelWeak',
         'S1OtsuLabelWeak', 'venv', '.venv', 'site-packages', '__pycache__'}

achados = {}
t0 = time.time()
for pasta, subpastas, arquivos in os.walk(MEUDRIVE, followlinks=True):
    subpastas[:] = [d for d in subpastas if not d.startswith('.') and d not in PULAR]
    for f in arquivos:
        if f in ALVOS:
            achados.setdefault(f, []).append(os.path.join(pasta, f))
print(f'Busca concluída em {time.time()-t0:.0f} s\n')

def primeiro(nome):
    lst = achados.get(nome) or []
    return lst[0] if lst else None

def mb(p):
    try: return f'{os.path.getsize(p)/1e6:,.1f} MB'
    except Exception: return '?'

for nome in sorted(ALVOS):
    for p in achados.get(nome, []):
        print(f'[ok] {nome:34s} {mb(p):>10s}  {p}')
    if nome not in achados:
        print(f'[--] {nome:34s} não encontrado' + ('' if nome in OBRIGATORIOS else ' (tudo bem: é baixado depois)'))
faltam = [n for n in OBRIGATORIOS if n not in achados]
assert not faltam, (f'Faltam {faltam}. Confira se as pastas paper_rs2024_* foram compartilhadas com esta conta '
                    'e se os atalhos estão em Mestrado/dados (passos 1 e 2 do topo).')
print('\nConta certa e arquivos encontrados.')

In [ ]:
# CÉLULA 2. Fixar o modelo: confere o hash e, se faltar, baixa a versão exata (cerca de 1,3 GB, só na primeira vez)
from huggingface_hub import hf_hub_download

REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11'
COMMIT = '918b9f140bb1783716664a2421ea3253d806017d'
ARQ_PESOS = 'Prithvi-EO-V2-300M-TL-Sen1Floods11.pt'
SHA_PESOS = '3675e9c2b52547de8ff8a19f4881c28573e6d4d2f0805d866f4fc48c1e517d60'
SHA_CONFIG = 'ecc9ba411c7e2ca0be70bb597e0aae45d866acaf1fdfd2f4b8ac652ee5eab0b0'
TEMP = '/content/hf_modelo'

def sha256(p, bloco=1 << 24):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(bloco), b''):
            h.update(b)
    return h.hexdigest()

# Pesos
caminho_pesos, h_pesos, origem = None, None, None
for f in sorted(os.listdir(FIXADO)):
    p = os.path.join(FIXADO, f)
    if f.endswith(('.pt', '.ckpt', '.pth', '.safetensors')) and os.path.getsize(p) > 100e6:
        h = sha256(p)
        print(f'{f}: sha256 {h}', '(igual ao do experimento)' if h == SHA_PESOS else '(diferente do experimento)')
        if h == SHA_PESOS:
            caminho_pesos, h_pesos, origem = p, h, 'já estava em modelo_fixado'
            break
if caminho_pesos is None:
    print(f'Baixando {ARQ_PESOS} no commit {COMMIT[:7]} (alguns minutos)...')
    local = hf_hub_download(REPO, ARQ_PESOS, revision=COMMIT, local_dir=TEMP)
    h = sha256(local)
    assert h == SHA_PESOS, f'O arquivo baixado tem hash {h}, diferente do esperado. Cole esta saída no Claude.'
    caminho_pesos = os.path.join(FIXADO, ARQ_PESOS)
    if os.path.exists(caminho_pesos):          # havia um arquivo com o mesmo nome e outro conteúdo: guarda com outro nome
        os.replace(caminho_pesos, caminho_pesos + '.outra_versao')
    print('Copiando para o Drive...')
    shutil.copy(local, caminho_pesos)
    h_pesos = sha256(caminho_pesos)
    assert h_pesos == SHA_PESOS, 'A cópia no Drive ficou diferente do arquivo baixado. Rode a célula de novo.'
    origem = f'baixado do Hugging Face no commit {COMMIT[:7]} em {time.strftime("%Y-%m-%d")}'

# Config
caminho_cfg = os.path.join(FIXADO, 'config.yaml')
if not (os.path.exists(caminho_cfg) and sha256(caminho_cfg) == SHA_CONFIG):
    if os.path.exists(caminho_cfg):
        os.replace(caminho_cfg, caminho_cfg + '.outra_versao')
    copia = next((p for p in achados.get('config.yaml', []) if p != caminho_cfg and sha256(p) == SHA_CONFIG), None)
    if copia:
        shutil.copy(copia, caminho_cfg)
        print('config.yaml copiado de', copia)
    else:
        shutil.copy(hf_hub_download(REPO, 'config.yaml', revision=COMMIT, local_dir=TEMP), caminho_cfg)
        print(f'config.yaml baixado no commit {COMMIT[:7]}')
h_cfg = sha256(caminho_cfg)

HASHES = {'repositorio_hf': REPO, 'commit_hf': COMMIT,
          'arquivo_pesos': caminho_pesos, 'sha256_pesos': h_pesos, 'pesos_iguais_ao_experimento': h_pesos == SHA_PESOS,
          'origem_pesos': origem,
          'arquivo_config': caminho_cfg, 'sha256_config': h_cfg, 'config_igual_ao_experimento': h_cfg == SHA_CONFIG}

# Registro do download original de maio (prova de qual versão gerou os escores)
for p in achados.get('config.yaml', []):
    meta = os.path.join(os.path.dirname(p), '.cache', 'huggingface', 'download', ARQ_PESOS + '.metadata')
    if os.path.exists(meta):
        try:
            commit_meta, etag_meta, quando = open(meta).read().split()[:3]
            HASHES['registro_download_original'] = {
                'arquivo': meta, 'commit': commit_meta, 'sha256': etag_meta,
                'data_utc': time.strftime('%Y-%m-%d %H:%M', time.gmtime(float(quando)))}
        except Exception as e:
            HASHES['registro_download_original'] = f'não consegui ler: {e}'
        break

print()
for k, v in HASHES.items():
    print(f'{k:28s} {v}')
assert HASHES['pesos_iguais_ao_experimento'] and HASHES['config_igual_ao_experimento']
print('\nModelo fixado e conferido.')

In [ ]:
# CÉLULA 3. Referência da fonte: calibrar na val, avaliar no test (bootstrap por chip, B = 2000)
ALFA, B, SEMENTE = 0.10, 2000, 0
NBINS = 6000                      # bins finos de confiança para o AUROC por histograma
scores = np.load(primeiro('cal_scores.npy'), mmap_mode='r')
labels = np.load(primeiro('cal_labels.npy'), mmap_mode='r')
tam = np.load(primeiro('cal_chip_sizes.npy')).astype(np.int64)
nomes = [x.decode() if isinstance(x, bytes) else str(x) for x in np.load(primeiro('cal_chip_names.npy'), allow_pickle=True)]
assert int(tam.sum()) == len(scores) == len(labels), 'tamanhos não batem; cole a saída no Claude'
assert len(nomes) == len(tam)

rx = re.compile(r'([A-Za-z]+_\d+)')
def chave(s):
    m = rx.search(os.path.basename(str(s))); return m.group(1) if m else None

import urllib.request
SPLITS = 'https://storage.googleapis.com/sen1floods11/v1.1/splits/flood_handlabeled/'
DESTINO_SPLITS = os.path.join(MESTRADO, 'dados', 'splits')
os.makedirs(DESTINO_SPLITS, exist_ok=True)

def achar_csv(nome):
    no_drive = os.path.join(DESTINO_SPLITS, nome)
    if os.path.exists(no_drive):
        return no_drive
    p = primeiro(nome)
    if p is not None and os.path.abspath(p) != no_drive:
        shutil.copy(p, no_drive)
        print(f'{nome} copiado para {DESTINO_SPLITS}')
        return no_drive
    urllib.request.urlretrieve(SPLITS + nome, no_drive)
    print(f'{nome} baixado do bucket oficial e salvo em {DESTINO_SPLITS}')
    return no_drive

def ids_csv(nome):
    linhas = open(achar_csv(nome)).read().strip().splitlines()
    return {chave(l.split(',')[0]) for l in linhas if chave(l.split(',')[0])}
VAL, TEST = ids_csv('flood_valid_data.csv'), ids_csv('flood_test_data.csv')
print(f'IDs nos CSVs oficiais: val {len(VAL)}, test {len(TEST)}')

part = np.array(['val' if chave(n) in VAL else 'test' if chave(n) in TEST else 'outro' for n in nomes])
print('chips por partição:', {k: int((part == k).sum()) for k in ['val', 'test', 'outro']})
assert (part == 'val').sum() > 0 and (part == 'test').sum() > 0, 'os nomes dos chips não casaram com os CSVs de split; cole a saída no Claude'
offs = np.concatenate([[0], np.cumsum(tam)])

# q-chapéu só com os pixels da val
s_val = np.concatenate([np.asarray(scores[offs[i]:offs[i+1]], dtype=np.float64)[np.isin(np.asarray(labels[offs[i]:offs[i+1]]), [0, 1])]
                        for i in np.where(part == 'val')[0]])
n = len(s_val)
qhat = float(np.quantile(s_val, min(1.0, np.ceil((n + 1) * (1 - ALFA)) / n), method='higher'))
tau = 1 - qhat
print(f'q-chapéu (val, {n:,} pixels) = {qhat:.4f}  ->  tau = {tau:.4f}')

ECE_BINS = np.linspace(0.5, 1.0, 16)
def bin_conf(msp):
    z = -np.log10(np.clip(1 - msp, 1e-12, 0.5))         # 0.30 (msp=0,5) a 12 (msp=1)
    return np.clip(((z - np.log10(2)) / (12 - np.log10(2)) * NBINS).astype(np.int64), 0, NBINS - 1)

def por_chip(i):
    s = np.asarray(scores[offs[i]:offs[i+1]], dtype=np.float64); y = np.asarray(labels[offs[i]:offs[i+1]])
    ok = np.isin(y, [0, 1]) & np.isfinite(s); s, y = s[ok], y[ok].astype(np.int64)
    err = s > 0.5; msp = np.maximum(s, 1 - s); pred = np.where(err, 1 - y, y)
    b = bin_conf(msp); eb = np.clip(np.digitize(msp, ECE_BINS) - 1, 0, 14)
    return dict(
        n=len(s), erro=int(err.sum()), cobre=int((s <= qhat).sum()),
        conf_err=int((err & (msp >= tau)).sum()),
        vazio=int(((s > qhat) & (s < 1 - qhat)).sum()),
        c11=int(((y == 1) & (pred == 1)).sum()), c10=int(((y == 1) & (pred == 0)).sum()),
        c01=int(((y == 0) & (pred == 1)).sum()), c00=int(((y == 0) & (pred == 0)).sum()),
        hc=np.bincount(b[~err], minlength=NBINS), he=np.bincount(b[err], minlength=NBINS),
        ece_n=np.bincount(eb, minlength=15), ece_conf=np.bincount(eb, weights=msp, minlength=15),
        ece_acc=np.bincount(eb, weights=(~err).astype(float), minlength=15))

t0 = time.time()
CH = [por_chip(i) for i in range(len(tam))]
print(f'contagens por chip prontas em {time.time()-t0:.0f} s')
ESC = ['n', 'erro', 'cobre', 'conf_err', 'vazio', 'c11', 'c10', 'c01', 'c00']
M = {k: np.array([c[k] for c in CH], dtype=np.float64) for k in ESC}
for k in ['hc', 'he', 'ece_n', 'ece_conf', 'ece_acc']:
    M[k] = np.stack([c[k] for c in CH]).astype(np.float64)

VAZIOS = [{'chip': nomes[i], 'particao': str(part[i])} for i in np.where(M['n'] == 0)[0]]
print(f'chips sem pixel válido (ficam fora do sorteio): {len(VAZIOS)}',
      [f"{v['chip'].replace('_LabelHand.tif', '')} ({v['particao']})" for v in VAZIOS])

def metricas(w):
    a = {k: w @ M[k] for k in M}
    N, E = a['n'], a['erro']
    hc, he = a['hc'], a['he']
    abaixo = np.cumsum(he, axis=-1) - he
    auroc = (hc * (abaixo + 0.5 * he)).sum(-1) / (hc.sum(-1) * he.sum(-1))
    iou1 = a['c11'] / (a['c11'] + a['c10'] + a['c01']); iou0 = a['c00'] / (a['c00'] + a['c01'] + a['c10'])
    ece = ((a['ece_conf'] - a['ece_acc'])).sum(-1) / a['ece_n'].sum(-1)   # ECE com sinal (+ = sobreconfiante)
    return {'taxa_erro': E / N, 'miou': (iou0 + iou1) / 2, 'cobertura': a['cobre'] / N, 'conjunto_vazio': a['vazio'] / N,
            'falha_silenciosa_conjunta': a['conf_err'] / N, 'erro_nao_sinalizado_condicional': a['conf_err'] / E,
            'auroc_e': auroc, 'ece_com_sinal': ece, 'checagem_soma': (a['cobre'] + a['vazio'] + a['conf_err']) / N}

rng = np.random.default_rng(SEMENTE)
def avalia(mask, rotulo):
    idx = np.where(mask & (M['n'] > 0))[0]          # chips vazios não entram no sorteio
    w0 = np.zeros(len(tam)); w0[idx] = 1
    ponto = metricas(w0)
    W = np.zeros((B, len(tam)))
    for b in range(B):
        W[b] += np.bincount(rng.choice(idx, len(idx)), minlength=len(tam))
    boot = metricas(W)
    res = {k: {'ponto': float(ponto[k]), 'ic95': [float(np.percentile(boot[k], 2.5)), float(np.percentile(boot[k], 97.5))]} for k in ponto}
    res['chips'] = int(len(idx)); res['pixels'] = int(w0 @ M['n'])
    print(f'\n== {rotulo} ({len(idx)} chips, {res["pixels"]:,} pixels) ==')
    for k in ponto:
        r = res[k]; fator = 1 if k in ('miou', 'auroc_e', 'checagem_soma') else 100; u = '' if fator == 1 else ' %'
        print(f'  {k:34s} {r["ponto"]*fator:8.3f}{u}   IC95 [{r["ic95"][0]*fator:.3f}; {r["ic95"][1]*fator:.3f}]')
    return res

REF = {'alfa': ALFA, 'qhat_val': qhat, 'tau': tau, 'pixels_val_calibracao': n,
       'test': avalia(part == 'test', 'FONTE, TEST (referência nova)'),
       'val': avalia(part == 'val', 'FONTE, VAL (só informativo: é o conjunto de calibração)'),
       'val_mais_test': avalia(part != 'outro', 'VAL + TEST juntos (conferência contra o doc 18)')}
print('\nConferência: os pontos do test devem repetir a rodada anterior (erro 2,456%, AUROC 0,937).')
print('Só os IC mudam um pouco, porque os chips vazios saíram do sorteio.')

In [ ]:
# CÉLULA 4. Precisão numérica: AUROC de detecção de erro na val em 32 e em 16 bits
from sklearn.metrics import roc_auc_score
iv = np.where(part == 'val')[0]
s32 = np.concatenate([np.asarray(scores[offs[i]:offs[i+1]])[np.isin(np.asarray(labels[offs[i]:offs[i+1]]), [0, 1])] for i in iv]).astype(np.float32)
correto = (s32 <= 0.5)
msp32 = np.maximum(s32, 1 - s32)
msp16 = msp32.astype(np.float16).astype(np.float32)
PREC = {'dtype_original_cal_scores': str(scores.dtype)}
for nome, m in [('32 bits', msp32), ('16 bits', msp16)]:
    a = roc_auc_score(correto, m)
    PREC[nome] = {'auroc_e': float(a), 'valores_distintos': int(len(np.unique(m))), 'fracao_msp_igual_a_1': float((m == 1).mean())}
    print(f'{nome}: AUROC_e = {a:.5f} | valores distintos = {PREC[nome]["valores_distintos"]:,} | MSP == 1: {PREC[nome]["fracao_msp_igual_a_1"]*100:.2f}%')
PREC['diferenca_auroc'] = PREC['32 bits']['auroc_e'] - PREC['16 bits']['auroc_e']
print(f'\nDiferença (32 - 16) = {PREC["diferenca_auroc"]:.5f}   | dtype original dos escores: {scores.dtype}')

In [ ]:
# CÉLULA 5. Bolívia no τ novo e diferença Bolívia − test, com IC (bootstrap por chip, B = 2000)
import itertools
with open(primeiro('fonte_registros.pkl'), 'rb') as f: FONTE_REG = [r for r in pickle.load(f) if r['n'] > 0]
with open(primeiro('bolivia_registros.pkl'), 'rb') as f: BOL_REG = [r for r in pickle.load(f) if r['n'] > 0]
ANTIGO = json.load(open(primeiro('resultados_bolivia_vs_fonte.json')))
NB = len(BOL_REG[0]['h']) // 2
idx_nome = {nm: i for i, nm in enumerate(nomes)}
base = lambda r: os.path.basename(str(r['nome']))
print(f'registros salvos: fonte {len(FONTE_REG)} chips, Bolívia {len(BOL_REG)} chips, {2*NB} posições por chip')

def bruto(i):
    s = np.asarray(scores[offs[i]:offs[i+1]], dtype=np.float64); y = np.asarray(labels[offs[i]:offs[i+1]]).astype(np.int64)
    ok = np.isin(y, [0, 1]) & np.isfinite(s)
    return s[ok], y[ok]

# 1) Qual metade do h guarda os erros (conferido com os escores brutos de 5 chips da fonte)
conferir = FONTE_REG[::max(1, len(FONTE_REG) // 5)][:5]
ordem = set()
for r in conferir:
    i = idx_nome[base(r)]
    assert int(r['n']) == int(tam[i]), f"número de pixels diferente em {base(r)}; cole a saída no Claude"
    s, y = bruto(i); ne = int((s > 0.5).sum()); h = np.asarray(r['h'])
    a, b = int(h[:NB].sum()), int(h[NB:].sum())
    ordem.add('erro_antes' if (a, b) == (ne, len(s) - ne) else 'acerto_antes' if (b, a) == (ne, len(s) - ne) else 'outra')
assert len(ordem) == 1 and 'outra' not in ordem, f'o h não está separado em erro e acerto ({ordem}); cole esta saída no Claude'
ORDEM_H = ordem.pop()
print('metade com os erros:', 'a primeira' if ORDEM_H == 'erro_antes' else 'a segunda')

# 2) Escala dos bins aprendida dos próprios dados da fonte: a confiança (MSP) bruta dos 175 chips,
#    ordenada, é fatiada pelas contagens acumuladas do h. Depois confere se isso reproduz o h de cada chip.
msp_e, msp_c = [], []
for r in FONTE_REG:
    s, _ = bruto(idx_nome[base(r)]); e = s > 0.5
    msp_e.append(s[e]); msp_c.append(1 - s[~e])
msp_e, msp_c = np.sort(np.concatenate(msp_e)), np.sort(np.concatenate(msp_c))

class Escala:
    def __init__(self, ordenado, cont):
        self.v, self.cont = ordenado, cont.astype(np.int64)
        self.sup = np.cumsum(self.cont); self.inf = self.sup - self.cont
        self.cheio = np.where(self.cont > 0)[0]
        ini = self.inf[self.cheio]
        self.ini = ordenado[ini]; self.fim = ordenado[self.sup[self.cheio] - 1]
        centro = np.add.reduceat(ordenado, ini) / self.cont[self.cheio]
        self.centro = np.interp(np.arange(NB), self.cheio, centro)   # MSP média de cada bin
    def frac_ge(self, t):                                            # fração de cada bin com MSP >= t
        n_lt = np.searchsorted(self.v, t, 'left')
        f = np.where(self.inf >= n_lt, 1.0, 0.0)
        c = self.cheio
        f[c] = np.clip((self.sup[c] - n_lt) / self.cont[c], 0, 1)
        return f
    def bin_de(self, x):
        return self.cheio[np.clip(np.searchsorted(self.ini, x, 'right') - 1, 0, None)]

def metades(r, sentido):                                             # (erros, acertos), em ordem crescente de MSP
    h = np.asarray(r['h'], dtype=np.float64)
    he, hc = (h[:NB], h[NB:]) if ORDEM_H == 'erro_antes' else (h[NB:], h[:NB])
    return (he if sentido[0] > 0 else he[::-1]), (hc if sentido[1] > 0 else hc[::-1])

melhor = None
for sentido in itertools.product([1, -1], repeat=2):
    HE_F = sum(metades(r, sentido)[0] for r in FONTE_REG); HC_F = sum(metades(r, sentido)[1] for r in FONTE_REG)
    assert HE_F.sum() == len(msp_e) and HC_F.sum() == len(msp_c), 'contagens totais não batem; cole a saída no Claude'
    ee, ec = Escala(msp_e, HE_F), Escala(msp_c, HC_F)
    d = []
    for r in conferir:
        s, _ = bruto(idx_nome[base(r)]); e = s > 0.5; he, hc = metades(r, sentido)
        d.append((np.abs(he - np.bincount(ee.bin_de(s[e]), minlength=NB)).sum() +
                  np.abs(hc - np.bincount(ec.bin_de(1 - s[~e]), minlength=NB)).sum()) / r['n'])
    if melhor is None or max(d) < melhor[0]:
        melhor = (max(d), sentido, ee, ec)
DIST_ESCALA, SENTIDO, ESC_E, ESC_C = melhor
print(f'reprodução do h chip a chip com a escala aprendida: diferença máxima {DIST_ESCALA:.1e}')
assert DIST_ESCALA < 5e-3, 'a escala aprendida não reproduz o h dos chips; cole esta saída no Claude'

# Erros e acertos usam os mesmos bins? (necessário para o AUROC por histograma)
lo_m = np.full(NB, np.inf); hi_m = np.full(NB, -np.inf)
for esc in (ESC_E, ESC_C):
    lo_m[esc.cheio] = np.minimum(lo_m[esc.cheio], esc.ini); hi_m[esc.cheio] = np.maximum(hi_m[esc.cheio], esc.fim)
usados = np.where(np.isfinite(lo_m))[0]
CRUZADOS = int((lo_m[usados][1:] < np.maximum.accumulate(hi_m[usados])[:-1]).sum())
print(f'bins em que erros e acertos se cruzam: {CRUZADOS} de {len(usados)} usados')
BORDAS = {int(k): [float(lo_m[k]), float(hi_m[k]), 0.5 + k * 0.5 / NB] for k in
          [0, 1, NB // 2, int((tau - 0.5) / 0.5 * NB), NB - 2, NB - 1] if np.isfinite(lo_m[k])}
print('bordas observadas de alguns bins [menor MSP, maior MSP, início se a escala fosse linear]:')
for k, v in BORDAS.items(): print(f'   bin {k:5d}: {v[0]:.7f}  {v[1]:.7f}  {v[2]:.7f}')

alvo = np.array([[CH[idx_nome[base(r)]][k] for k in ('c00', 'c01', 'c10', 'c11')] for r in conferir], dtype=np.float64)
conf_salva = np.array([np.asarray(r['conf'], dtype=np.float64) for r in conferir])
PERM = next((list(p) for p in itertools.permutations(range(4))
             if np.allclose(conf_salva[:, list(p)], alvo, rtol=1e-3, atol=2)), None)
print('ordem da matriz de confusão salva:', PERM if PERM else 'não identificada (o mIoU fica de fora)')

# 3) Métricas a partir dos histogramas
def matrizes(regs):
    par = [metades(r, SENTIDO) for r in regs]
    return np.stack([p[0] for p in par]), np.stack([p[1] for p in par])
def confs(regs): return None if PERM is None else np.array([np.asarray(r['conf'], dtype=np.float64)[PERM] for r in regs])

def auroc(HE, HC, W, bloco=250):
    W2 = np.atleast_2d(W); out = []
    for i in range(0, len(W2), bloco):
        he = W2[i:i + bloco] @ HE; hc = W2[i:i + bloco] @ HC
        abaixo = np.cumsum(he, axis=1) - he
        out.append((hc * (abaixo + 0.5 * he)).sum(1) / (hc.sum(1) * he.sum(1)))
    out = np.concatenate(out)
    return out if np.ndim(W) == 2 else out[0]

def met(HE, HC, conf, ta, W):
    fe, fc = ESC_E.frac_ge(ta), ESC_C.frac_ge(ta)
    E = W @ HE.sum(1); N = E + W @ HC.sum(1)
    cobre = W @ (HC @ fc)                     # acerto com MSP >= tau  <=>  s <= q-chapéu
    conf_err = W @ (HE @ fe)                  # erro com MSP >= tau
    soma_msp = W @ (HE @ ESC_E.centro + HC @ ESC_C.centro)
    out = {'taxa_erro': E / N, 'cobertura': cobre / N, 'conjunto_vazio': (N - cobre - conf_err) / N,
           'falha_silenciosa_conjunta': conf_err / N, 'erro_nao_sinalizado_condicional': conf_err / E,
           'auroc_e': auroc(HE, HC, W), 'ece_com_sinal': soma_msp / N - (1 - E / N)}
    if conf is not None:
        c = W @ conf
        c00, c01, c10, c11 = c[..., 0], c[..., 1], c[..., 2], c[..., 3]
        out['miou'] = (c11 / (c11 + c10 + c01) + c00 / (c00 + c01 + c10)) / 2
    return out

part_de = dict(zip(nomes, part))
REG_T = [r for r in FONTE_REG if part_de.get(base(r)) == 'test']
(HE_B, HC_B), (HE_T, HC_T), (HE_A, HC_A) = matrizes(BOL_REG), matrizes(REG_T), matrizes(FONTE_REG)

# 4) Conferência: no τ antigo, os registros devem reproduzir o doc 18
T_OLD = float(ANTIGO['tau'])
MAPA = {'taxa_erro': 'taxa_erro', 'miou': 'miou', 'cobertura': 'cobertura', 'conj_vazio': 'conjunto_vazio',
        'erro_confiante_px': 'falha_silenciosa_conjunta', 'falha_silenciosa': 'erro_nao_sinalizado_condicional',
        'auroc_erro': 'auroc_e'}
MAPA.update({k: 'ece_com_sinal' for k in ANTIGO['metricas']                      # só o ECE com sinal; o doc 18 guarda também o sem sinal
             if 'ece' in k.lower() and ('sinal' in k.lower() or 'sign' in k.lower())})
TOL = {'auroc_e': 0.005, 'miou': 0.005, 'erro_nao_sinalizado_condicional': 0.01}
print(f'\nConferência com o doc 18 (τ antigo = {T_OLD:.4f}):')
CONF18, problemas = {}, []
for grupo, HE, HC, regs in [('fonte', HE_A, HC_A, FONTE_REG), ('bolivia', HE_B, HC_B, BOL_REG)]:
    m = met(HE, HC, confs(regs), T_OLD, np.ones(len(HE)))
    for k_ant, k_novo in MAPA.items():
        if k_ant not in ANTIGO['metricas'] or k_novo not in m or grupo not in ANTIGO['metricas'][k_ant]:
            continue
        antes, agora = float(ANTIGO['metricas'][k_ant][grupo]['ponto']), float(m[k_novo])
        if k_novo == 'ece_com_sinal' and abs(antes) > 0.2:
            agora *= 100                              # o doc 18 guardou o ECE em pontos percentuais
        ok = abs(antes - agora) <= TOL.get(k_novo, 0.002)
        CONF18[f'{grupo}.{k_novo}'] = {'doc18': antes, 'agora': agora, 'confere': ok}
        if not ok: problemas.append(f'{grupo}.{k_novo}')
        print(f'  {grupo:8s} {k_novo:34s} doc 18 {antes:9.5f} | agora {agora:9.5f}  {"confere" if ok else "DIFERENTE"}')
print('Tudo confere.' if not problemas else f'Atenção, não conferem: {problemas}. Cole esta saída no Claude.')

# 5) No τ novo (calibrado só na val): Bolívia, test e diferença
rng = np.random.default_rng(1)
pesos_boot = lambda k: np.stack([np.bincount(rng.choice(k, k), minlength=k) for _ in range(B)]).astype(np.float64)
WB, WT = pesos_boot(len(HE_B)), pesos_boot(len(HE_T))
cb, ct = confs(BOL_REG), confs(REG_T)
pb, bb = met(HE_B, HC_B, cb, tau, np.ones(len(HE_B))), met(HE_B, HC_B, cb, tau, WB)
pt, bt = met(HE_T, HC_T, ct, tau, np.ones(len(HE_T))), met(HE_T, HC_T, ct, tau, WT)
ic = lambda x: [float(np.nanpercentile(x, 2.5)), float(np.nanpercentile(x, 97.5))]

ORDEM = [k for k in ['taxa_erro', 'miou', 'cobertura', 'conjunto_vazio', 'falha_silenciosa_conjunta',
                     'erro_nao_sinalizado_condicional', 'auroc_e', 'ece_com_sinal'] if k in pb]
fmt = lambda k, v: f'{v:.3f}' if k in ('miou', 'auroc_e') else f'{v*100:.2f}'
col = lambda k, p, i: f'{fmt(k, p)} [{fmt(k, i[0])}; {fmt(k, i[1])}]'
print(f'\nτ novo = {tau:.4f}. Bolívia {len(HE_B)} chips, test {len(HE_T)} chips. Taxas em %, ECE em pontos percentuais.\n')
print(f'  {"métrica":34s}{"Bolívia [IC95]":26s}{"test [IC95]":26s}{"Bolívia − test [IC95]":28s}')
for k in ORDEM:
    icd = ic(bb[k] - bt[k])
    fora = 'IC exclui 0' if icd[0] > 0 or icd[1] < 0 else 'IC inclui 0'
    print(f'  {k:34s}{col(k, pb[k], ic(bb[k])):26s}{col(k, pt[k], ic(bt[k])):26s}{col(k, pb[k] - pt[k], icd):28s}{fora}')

print('\nConferência: test pelos histogramas x test exato da célula 3 (pontos):')
for k in ORDEM:
    print(f'  {k:34s} histograma {fmt(k, pt[k]):>7s} | exato {fmt(k, REF["test"][k]["ponto"]):>7s}')

BOL = {'ordem_h': ORDEM_H, 'sentido_metades': list(SENTIDO), 'bins_por_metade': NB,
       'escala': {'metodo': 'bins aprendidos da MSP bruta da fonte, ordenada e fatiada pelas contagens acumuladas do h',
                  'reproducao_chip_a_chip_dif_max': DIST_ESCALA, 'bins_cruzados': CRUZADOS, 'bins_usados': len(usados),
                  'bordas_amostra': BORDAS},
       'ordem_conf': PERM, 'qhat': qhat, 'tau': tau, 'B': B, 'semente': 1,
       'chips_bolivia': len(HE_B), 'chips_test': len(HE_T),
       'conferencia_doc18': CONF18, 'conferencia_doc18_ok': not problemas,
       'bolivia': {k: {'ponto': float(pb[k]), 'ic95': ic(bb[k])} for k in ORDEM},
       'test_pelos_histogramas': {k: {'ponto': float(pt[k]), 'ic95': ic(bt[k])} for k in ORDEM},
       'diferenca_bolivia_menos_test': {k: {'ponto': float(pb[k] - pt[k]), 'ic95': ic(bb[k] - bt[k])} for k in ORDEM}}

In [ ]:
# CÉLULA 6. Salvar no Drive do mestrado
saida = os.path.join(MESTRADO, 'resultados', 'semana1')
os.makedirs(saida, exist_ok=True)

def para_json(o):
    if isinstance(o, np.generic): return o.item()
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

tudo = {'data': time.strftime('%Y-%m-%d %H:%M'), 'hashes': HASHES, 'referencia_fonte': REF, 'chips_vazios': VAZIOS,
        'precisao': PREC, 'bolivia_tau_novo': BOL,
        'chips_val': [c for c, p in zip(nomes, part) if p == 'val'],
        'chips_test': [c for c, p in zip(nomes, part) if p == 'test']}
with open(os.path.join(saida, 'semana1_resultados.json'), 'w') as f:
    json.dump(tudo, f, indent=2, ensure_ascii=False, default=para_json)
np.savez_compressed(os.path.join(saida, 'fonte_contagens_por_chip.npz'), nomes=np.array(nomes), particao=part, qhat=qhat, **M)
print('Salvo em', saida)
print('\nPronto. Anexe no Claude o arquivo semana1_resultados.json dessa pasta.')